# Análise de Clusterização com K-Means

**Cientista de Dados:** Análise não supervisionada usando o algoritmo K-Means

**Base utilizada:** Mall Customers Segmentation Dataset — base clássica para segmentação de clientes contendo informações demográficas e de consumo.

## Objetivos
1. Realizar Análise Exploratória dos Dados (EDA)
2. Preparar os dados (escalonamento)
3. Determinar o número ótimo de clusters (Método do Cotovelo + Silhouette)
4. Aplicar o K-Means
5. Visualizar e interpretar os agrupamentos
6. Gerar insights de negócio

> Para executar: abra este notebook no Google Colab (`File > Upload notebook`) e execute as células em ordem (`Runtime > Run all`).

## 1. Importação das bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento da base de dados

Usaremos a base `Mall_Customers.csv`. Você pode:
- Fazer upload no Colab (`from google.colab import files`)
- Ou usar o link público abaixo

Caso o link falhe, geramos um dataset sintético equivalente para fins didáticos.

In [ ]:
URL = 'https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv'

try:
    df = pd.read_csv(URL)
    print('Base carregada do repositório público.')
except Exception as e:
    print(f'Falha ao baixar ({e}). Gerando dataset sintético equivalente...')
    n = 200
    df = pd.DataFrame({
        'CustomerID': np.arange(1, n + 1),
        'Gender': np.random.choice(['Male', 'Female'], size=n),
        'Age': np.random.randint(18, 70, size=n),
        'Annual Income (k$)': np.random.randint(15, 140, size=n),
        'Spending Score (1-100)': np.random.randint(1, 100, size=n),
    })

df.head()

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
print('Dimensões:', df.shape)
print('\nTipos de dados:')
print(df.dtypes)
print('\nValores nulos:')
print(df.isnull().sum())
print('\nEstatísticas descritivas:')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Age'], kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribuição de Idade')
sns.histplot(df['Annual Income (k$)'], kde=True, color='seagreen', ax=axes[1])
axes[1].set_title('Distribuição de Renda Anual (k$)')
sns.histplot(df['Spending Score (1-100)'], kde=True, color='salmon', ax=axes[2])
axes[2].set_title('Distribuição do Spending Score')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Gender', data=df, palette='pastel', ax=axes[0])
axes[0].set_title('Distribuição por Gênero')

sns.scatterplot(
    x='Annual Income (k$)', y='Spending Score (1-100)',
    hue='Gender', data=df, palette='Set1', s=80, ax=axes[1]
)
axes[1].set_title('Renda x Spending Score')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.select_dtypes(include=np.number).drop(columns=['CustomerID']).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Matriz de Correlação')
plt.show()

## 4. Pré-processamento

Selecionamos as variáveis numéricas relevantes e aplicamos `StandardScaler`, pois o K-Means é sensível à escala (usa distâncias euclidianas).

In [ ]:
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=features).describe().round(3)

## 5. Definindo o K ótimo

Combinamos duas técnicas:
- **Método do Cotovelo (Elbow):** observa a inércia (WCSS) em função de K.
- **Coeficiente de Silhouette:** mede o quão bem os pontos foram alocados aos clusters (varia de -1 a 1).

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(K_range, inertias, marker='o', color='steelblue')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].set_title('Método do Cotovelo')
axes[0].grid(True)

axes[1].plot(K_range, silhouettes, marker='o', color='seagreen')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Coeficiente de Silhouette')
axes[1].grid(True)
plt.tight_layout()
plt.show()

best_k = K_range[int(np.argmax(silhouettes))]
print(f'K sugerido pelo Silhouette: {best_k} (score = {max(silhouettes):.3f})')

## 6. Aplicando o K-Means

Para a base Mall Customers o K=5 é tradicionalmente o melhor (cotovelo claro e bom silhouette).

In [ ]:
K = 5
kmeans = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)
centroides_df = pd.DataFrame(centers_original, columns=features)
centroides_df.index.name = 'Cluster'
print('Centroides (escala original):')
centroides_df.round(2)

## 7. Visualização dos clusters

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='Annual Income (k$)', y='Spending Score (1-100)',
    hue='Cluster', data=df, palette='tab10', s=100, edgecolor='black'
)
plt.scatter(
    centroides_df['Annual Income (k$)'], centroides_df['Spending Score (1-100)'],
    s=300, c='black', marker='X', label='Centroides'
)
plt.title('Clusters: Renda Anual vs Spending Score')
plt.legend()
plt.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=df['Cluster'], palette='tab10', s=100, edgecolor='black'
)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)')
plt.title('Clusters em 2D (PCA)')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
scat = ax.scatter(
    df['Age'], df['Annual Income (k$)'], df['Spending Score (1-100)'],
    c=df['Cluster'], cmap='tab10', s=60, edgecolor='black'
)
ax.set_xlabel('Idade')
ax.set_ylabel('Renda Anual (k$)')
ax.set_zlabel('Spending Score')
ax.set_title('Visualização 3D dos clusters')
plt.show()

## 8. Interpretação dos clusters

In [ ]:
perfil = df.groupby('Cluster').agg(
    qtd_clientes=('CustomerID', 'count'),
    idade_media=('Age', 'mean'),
    renda_media=('Annual Income (k$)', 'mean'),
    score_medio=('Spending Score (1-100)', 'mean')
).round(2)
perfil

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(x='Cluster', y='Age', data=df, palette='tab10', ax=axes[0])
axes[0].set_title('Idade por cluster')
sns.boxplot(x='Cluster', y='Annual Income (k$)', data=df, palette='tab10', ax=axes[1])
axes[1].set_title('Renda por cluster')
sns.boxplot(x='Cluster', y='Spending Score (1-100)', data=df, palette='tab10', ax=axes[2])
axes[2].set_title('Spending Score por cluster')
plt.tight_layout()
plt.show()

## 9. Insights e recomendações de negócio

Com base nos perfis encontrados, costumamos identificar nos dados Mall Customers cinco segmentos típicos:

| Cluster | Perfil | Estratégia sugerida |
|---|---|---|
| Alta renda + Alto consumo | **Clientes-alvo / VIP** | Programa de fidelidade premium, cross-sell |
| Alta renda + Baixo consumo | **Cautelosos** | Campanhas para destravar consumo (cupons, ofertas exclusivas) |
| Baixa renda + Alto consumo | **Engajados** | Cuidar da retenção, oferecer parcelamento |
| Baixa renda + Baixo consumo | **Inativos / desinteressados** | Reativação leve, baixo investimento |
| Renda + consumo médios | **Clientes padrão** | Cross-sell genérico, foco em volume |

## 10. Conclusão
- O K-Means segmentou os clientes em grupos com perfis claros de **idade**, **renda** e **comportamento de consumo**.
- A combinação **Elbow + Silhouette** confirmou K=5 como uma boa escolha.
- Os clusters viabilizam ações **direcionadas de marketing**, otimizando ROI de campanhas.

### Próximos passos
- Testar **DBSCAN** ou **Hierarchical Clustering** para comparação.
- Adicionar variáveis comportamentais (frequência, ticket médio) para clusters mais ricos.
- Validar a estabilidade dos clusters em janelas temporais distintas.